In [1]:
# 代码作用：读取MES、LIMS数据汇总为B糖煮制(班报) v1
# 核心逻辑：全局累计合格率（按榨季独立累计，跨日期连续累计）
# 2026/01/21 修正：严格按榨季分区计算累计合格率
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType, DateType,DoubleType
# 关键：明确导入Spark函数，避免和Python内置函数混淆
from pyspark.sql.functions import (
    current_timestamp, split, lit, col, row_number, sum as spark_sum, 
    round as spark_round, avg, when, to_date, count, coalesce
)
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()

# ===================== 1. 读取MES数据（逻辑不变） =====================
mes_sql = """
WITH ranked_data AS (
    SELECT 
        T1.season,
        T1.factory,
        T1.hand_date,
        T1.team,
        T2.project,
        CASE WHEN T2.value REGEXP '^[0-9.]+$' THEN CAST(T2.value AS DECIMAL(10,2)) ELSE NULL END AS value,
        ROW_NUMBER() OVER (
            PARTITION BY T1.season, T1.factory, T1.hand_date, T1.team, T2.project
            ORDER BY T1.hand_time ASC
        ) AS rn
    FROM dwd.dwd_mes_pro_shift_change_record_f_1h T1 
    INNER JOIN dwd.dwd_mes_pro_shift_change_detail_f_1h T2 
        ON T1.hand_code = T2.record_code
    WHERE T2.project IN ("当班原料用量(m³)：","当班放糖量(m³)：","当班种子用量(m³)：") 
      AND T1.flag_deleted = 0
),
unique_data AS (
    SELECT 
        season,
        factory,
        hand_date,
        team,
        project,
        value
    FROM ranked_data
    WHERE rn = 1
)
SELECT 
    season,
    factory,
    hand_date,
    team,
    MAX(CASE WHEN project = '当班原料用量(m³)：' THEN value END) AS raw_material_vol_m3,
    MAX(CASE WHEN project = '当班放糖量(m³)：' THEN value END) AS sugar_output_vol_m3,
    MAX(CASE WHEN project = '当班种子用量(m³)：' THEN value END) AS seed_vol_m3
FROM unique_data
GROUP BY season, factory, hand_date, team
ORDER BY season, factory, hand_date, team;
"""
mes_df = spark.sql(mes_sql)

# ===================== 2. 读取LIMS数据（最终修正版，无多余字段） =====================
lims_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        test_name1,
        ord_up_ver,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_sample_name LIKE '%乙膏%'
        AND test_name1 IN ('锤度', '视纯度')
),
paste_b_data AS (
    SELECT 
        company,
        crushing_season,
        record_date AS paste_record_date,
        work_shift AS paste_work_shift,
        MIN(ord_up_ver) AS min_ord_up_ver,
        MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS paste_b_brix,
        MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS paste_b_apparent_purity
    FROM raw_data
    WHERE ord_sample_name LIKE '%乙膏%'
    GROUP BY company, crushing_season, record_date, work_shift
),
ranked_paste_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY company, crushing_season, paste_record_date, paste_work_shift
            ORDER BY min_ord_up_ver ASC
        ) AS rn
    FROM paste_b_data
)
SELECT 
    company,
    crushing_season,
    paste_record_date,
    paste_work_shift,
    paste_b_brix,
    paste_b_apparent_purity
FROM ranked_paste_data
WHERE rn = 1
"""
lims_df = spark.sql(lims_sql)

# ===================== 3. 读取其他LIMS辅助数据（逻辑不变）=====================
# 读取lims班报糖膏数据
lims_ctj_sql = """
SELECT 
    season,
    gongsidm,
    rq,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        ELSE banbiedm
    END AS banbiedm,
    brix_jym,
    brix_yt,
    brix_ytg,
    apparent_purity_ytg,
    apparent_purity_yym
FROM dwd.dwd_lims_sugar_raw_molasses_batch_report_f_1h WHERE gongsidm = 'FNR';
"""
lims_ctj_df = spark.sql(lims_ctj_sql)
lims_ctj_df = lims_ctj_df.withColumnRenamed("season", "ctj_season")

# 读取lims班报实绩数据
lims_bbsj_sql = """
SELECT 
season,
tb_gongsidm,
tb_rq,
CASE 
WHEN tb_banbiedm = '01' THEN '甲班'
WHEN tb_banbiedm = '02' THEN '乙班'
WHEN tb_banbiedm = '03' THEN '丙班'
ELSE '未知班别' 
END AS banbie_name,
sj_zhazheliang
FROM dwd.dwd_lims_batch_report_actual_data_f_1h WHERE tb_gongsidm = 'FNR';
"""
lims_bbsj_df = spark.sql(lims_bbsj_sql)
lims_bbsj_df = lims_bbsj_df.withColumnRenamed("season", "bbsj_season") 

# ===================== 4. 数据预处理：统一日期+关联基础数据 =====================
# 统一日期类型
lims_bbsj_df = lims_bbsj_df.withColumn("tb_rq", to_date(col("tb_rq")))
mes_df = mes_df.withColumn("hand_date", to_date(col("hand_date")))
lims_ctj_df = lims_ctj_df.withColumn("rq", to_date(col("rq")))
lims_df = lims_df.withColumn("paste_record_date", to_date(col("paste_record_date")))

# 关联MES+LIMS班报实绩
mes_lims_df_v1 = lims_bbsj_df.join(
    mes_df,
    on=[
        lims_bbsj_df["bbsj_season"] == mes_df["season"],
        lims_bbsj_df["tb_gongsidm"] == mes_df["factory"],
        lims_bbsj_df["tb_rq"] == mes_df["hand_date"],
        lims_bbsj_df["banbie_name"] == mes_df["team"]
    ],
    how="left"
).select(
    col("bbsj_season"),
    col("tb_gongsidm"),
    col("tb_rq"),
    col("banbie_name"),
    col("sj_zhazheliang"),
    col("raw_material_vol_m3"),
    col("sugar_output_vol_m3"),
    col("seed_vol_m3")
)

# 关联糖膏基础数据
mes_lims_df_v2 = mes_lims_df_v1.join(
    lims_ctj_df,
    on=[
        mes_lims_df_v1["bbsj_season"] == lims_ctj_df["ctj_season"],
        mes_lims_df_v1["tb_gongsidm"] == lims_ctj_df["gongsidm"],
        mes_lims_df_v1["tb_rq"] == lims_ctj_df["rq"],
        mes_lims_df_v1["banbie_name"] == lims_ctj_df["banbiedm"]
    ],
    how="left"
)

# 新增基础比例字段（逻辑不变）
mes_lims_df_v2 = mes_lims_df_v2.withColumn(
    "seed_raw_material_ratio",
    when(
        (col("seed_vol_m3").isNotNull()) & (col("raw_material_vol_m3").isNotNull()) & (col("raw_material_vol_m3") != 0),
        col("seed_vol_m3") / col("raw_material_vol_m3")
    ).otherwise(None)
).withColumn(
    "sugar_output_cane_ratio",
    when(
        (col("sugar_output_vol_m3").isNotNull()) & (col("sj_zhazheliang").isNotNull()) & (col("sj_zhazheliang") != 0),
        col("sugar_output_vol_m3") / col("sj_zhazheliang")
    ).otherwise(None)
).withColumn(
    "sugar_output_solid_solute_t",
    when(
        (col("sugar_output_vol_m3").isNotNull()) & (col("brix_ytg").isNotNull()),
        col("sugar_output_vol_m3") * 1.54 * col("brix_ytg") * 0.001
    ).otherwise(None)
).withColumn(
    "apparent_purity_diff",
    when(
        (col("apparent_purity_ytg").isNotNull()) & (col("apparent_purity_yym").isNotNull()),
        col("apparent_purity_ytg") - col("apparent_purity_yym")
    ).otherwise(None)
)

# 筛选基础字段
keep_columns = [
    "bbsj_season", "tb_gongsidm", "tb_rq", "banbie_name",
    "sj_zhazheliang", "raw_material_vol_m3", "brix_jym", "seed_vol_m3",
    "brix_yt", "seed_raw_material_ratio", "sugar_output_vol_m3",
    "sugar_output_cane_ratio", "brix_ytg", "apparent_purity_ytg",
    "apparent_purity_yym", "sugar_output_solid_solute_t", "apparent_purity_diff"
]
mes_lims_df_v2 = mes_lims_df_v2.select(*keep_columns)

# ===================== 5. 核心逻辑：按榨季独立计算全局累计合格率 =====================
# 5.1 预处理LIMS数据，标记合格状态
lims_df_pre = lims_df.withColumn(
    "is_brix_qualified",
    when(
        (col("paste_b_brix").isNotNull()) & (col("paste_b_brix") >= 97) & (col("paste_b_brix") <= 101),
        lit(1)
    ).otherwise(lit(0))
).withColumn(
    "is_purity_qualified",
    when(
        (col("paste_b_apparent_purity").isNotNull()) & (col("paste_b_apparent_purity") >= 65) & (col("paste_b_apparent_purity") <= 75),
        lit(1)
    ).otherwise(lit(0))
)

# 5.2 定义全局窗口：【关键修正】严格按榨季+公司分区，确保不同榨季数据独立累计
# 优先级：榨季 > 公司 > 日期 > 班次
lims_df_pre = lims_df_pre.withColumn(
    "shift_sort",
    when(col("paste_work_shift") == "甲班", 1)
    .when(col("paste_work_shift") == "乙班", 2)
    .when(col("paste_work_shift") == "丙班", 3)
    .otherwise(99)
)

# 核心窗口定义：PARTITION BY crushing_season (榨季) 是关键，确保跨榨季不累计
global_window = Window.partitionBy("crushing_season", "company")\
    .orderBy("paste_record_date", "shift_sort")\
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 5.3 计算按榨季独立的全局累计值
lims_df_cumulative = lims_df_pre.withColumn(
    "global_row_num", row_number().over(global_window)  # 每个榨季内的累计行数
).withColumn(
    "cum_brix_qualified", spark_sum("is_brix_qualified").over(global_window)  # 榨季内累计合格数
).withColumn(
    "cum_purity_qualified", spark_sum("is_purity_qualified").over(global_window)  # 榨季内累计合格数
).withColumn(
    # 避免除以0，用coalesce处理空值
    "b_paste_brix_qualify_rate", 
    spark_round(
        coalesce(col("cum_brix_qualified") / col("global_row_num"), lit(0)), 
        3
    )
).withColumn(
    "b_paste_purity_qualify_rate", 
    spark_round(
        coalesce(col("cum_purity_qualified") / col("global_row_num"), lit(0)), 
        3
    )
)

# 5.4 提取合格率数据（仅保留关联字段+合格率）
qualify_rate_df = lims_df_cumulative.select(
    col("company"),
    col("crushing_season"),
    col("paste_record_date"),
    col("paste_work_shift"),
    col("b_paste_brix_qualify_rate"),
    col("b_paste_purity_qualify_rate")
)

# ===================== 6. 关联合格率到主数据 =====================
mes_lims_df_v3 = mes_lims_df_v2.join(
    qualify_rate_df,
    on=[
        mes_lims_df_v2["tb_gongsidm"] == qualify_rate_df["company"],
        mes_lims_df_v2["bbsj_season"] == qualify_rate_df["crushing_season"],  # 确保榨季严格匹配
        mes_lims_df_v2["tb_rq"] == qualify_rate_df["paste_record_date"],
        mes_lims_df_v2["banbie_name"] == qualify_rate_df["paste_work_shift"]
    ],
    how="left"
).select(
    mes_lims_df_v2["*"],
    col("b_paste_brix_qualify_rate"),
    col("b_paste_purity_qualify_rate")
)

# 补充源标识和更新时间
mes_lims_df_final = mes_lims_df_v3.withColumn("source_flg", lit("MES+LIMS")) \
    .withColumn("dw_update_time", current_timestamp())


# 合并主数据
mes_lims_df_final_with_daily = mes_lims_df_final

# ===================== 8. 写入Hive表 =====================
target_table = "app.app_mes_b_syrup_continuous_cook_control_params_team_stats_f_1h"

# 定义目标Schema
target_schema = StructType([
    StructField("bbsj_season", StringType(), True),
    StructField("tb_gongsidm", StringType(), True),
    StructField("tb_rq", DateType(), True),
    StructField("banbie_name", StringType(), True),
    StructField("sj_zhazheliang", DecimalType(30,10), True),
    StructField("raw_material_vol_m3", DecimalType(10,2), True),
    StructField("brix_jym", DecimalType(32,10), True),
    StructField("seed_vol_m3", DecimalType(10,2), True),
    StructField("brix_yt", DecimalType(32,10), True),
    StructField("seed_raw_material_ratio", DecimalType(23,13), True),
    StructField("sugar_output_vol_m3", DecimalType(10,2), True),
    StructField("sugar_output_cane_ratio", DecimalType(38,20), True),
    StructField("brix_ytg", DecimalType(32,10), True),
    StructField("apparent_purity_ytg", DecimalType(32,10), True),
    StructField("apparent_purity_yym", DecimalType(32,10), True),
    StructField("sugar_output_solid_solute_t", DoubleType(), True),
    StructField("apparent_purity_diff", DecimalType(33,10), True),
    StructField("b_paste_brix_qualify_rate", DoubleType(), True),
    StructField("b_paste_purity_qualify_rate", DoubleType(), True),
    StructField("source_flg", StringType(), True),
    StructField("dw_update_time", TimestampType(), True)
])

# 转换Schema并写入
df_to_write = spark.createDataFrame(mes_lims_df_final_with_daily.rdd, schema=target_schema)
df_to_write.write \
    .mode("overwrite") \
    .format("hive") \
    .saveAsTable(target_table)

# 添加表注释
spark.sql(f"""
    ALTER TABLE {target_table} 
    SET TBLPROPERTIES ('comment' = 'B糖煮制(班报) v1，按榨季独立计算全局累计合格率（跨日期连续累计）')
""")

# 字段注释
field_comments = {
    "bbsj_season": "榨季",
    "tb_gongsidm": "公司代码",
    "tb_rq": "日期",
    "banbie_name": "班别",
    "sj_zhazheliang": "榨蔗量",
    "raw_material_vol_m3": "A蜜量M3",
    "brix_jym": "A蜜锤度BX",
    "seed_vol_m3": "B种量M3",
    "brix_yt": "B种锤度BX",
    "seed_raw_material_ratio": "B种与A蜜比例",
    "sugar_output_vol_m3": "B糖膏放糖立方数M3",
    "sugar_output_cane_ratio": "B糖膏立方数对蔗比",
    "brix_ytg": "B糖膏锤度Bx 97～101",
    "apparent_purity_ytg": "B糖膏纯度AP 65～75",
    "apparent_purity_yym": "乙原液视纯度",
    "sugar_output_solid_solute_t": "B糖膏固溶物量T",
    "apparent_purity_diff": "B糖膏蜜纯度差",
    "b_paste_brix_qualify_rate": "B糖膏锤度按榨季累计合格率",
    "b_paste_purity_qualify_rate": "B糖膏纯度按榨季累计合格率",
    "source_flg": "数据来源标识",
    "dw_update_time": "数仓更新时间"
}

# 自动转换字段类型并添加注释
def get_hive_type_str(field):
    data_type = field.dataType
    type_name = type(data_type).__name__
    if type_name == "StringType": return "STRING"
    elif type_name == "DateType": return "DATE"
    elif type_name == "TimestampType": return "TIMESTAMP"
    elif type_name == "DoubleType": return "DOUBLE"
    elif type_name == "DecimalType": return f"DECIMAL({data_type.precision},{data_type.scale})"
    else: return "STRING"

field_type_map = {f.name: get_hive_type_str(f) for f in target_schema.fields}
for field, comment in field_comments.items():
    spark.sql(f"""
        ALTER TABLE {target_table} 
        CHANGE COLUMN {field} {field} {field_type_map.get(field, "STRING")} COMMENT '{comment}'
    """)

# 输出结果
print(f"数据已写入表：{target_table}")
# 停止Spark
spark.stop()

Spark 启动中...
数据已写入表：app.app_mes_b_syrup_continuous_cook_control_params_team_stats_f_1h
